# Figure 1: Cortical Functional Gradients

This notebook reproduces **Figure 1** from the manuscript, with one code cell per subpanel (A–G). It assumes precomputed timecourses and connectivity matrices are available on disk, as configured via `config-results.json`.


## Config and imports

This cell loads the configuration file and imports all required libraries and utility functions.


In [ ]:
from utils import *

# Load analysis config
params = read_config('config-results.json')

# Schaefer SMC atlas
n_rois = 400
schaefer_dataset = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, resolution_mm=2)
schaefer_atlas = schaefer_dataset.maps
schaefer_labels = np.asarray(schaefer_dataset.labels, dtype=str)
smc_labels = []
for i,label in enumerate(schaefer_labels):
    if 'SomMot' in label:
        smc_labels.append(label)
smc_labels = pd.Series(smc_labels)

# Custom spinal cord atlas
sc_data = load_img(params["custom_sc_atlas"]).get_fdata()
sc_labels = open(params["custom_sc_labels"], 'r').read().splitlines()

## ROI selection and corticospinal FC

Load FC matrices, compute group-level FC, and build the SMC-only FC matrix used throughout Figure 1.


In [ ]:
# ------------------------------------------------------------------
# Load precomputed subject-level ROI-restricted FC matrices
# from params["save_fc_mats"] instead of recomputing FC upstream
# ------------------------------------------------------------------
fc_files = sorted(glob.glob(os.path.join(params["save_fc_mats"], "*.csv")))

if len(fc_files) == 0:
    raise FileNotFoundError(
        f"No FC csv files found in: {params['save_fc_mats']}"
    )

subFC_mats = []
sub_rois = None

for fpath in fc_files:
    df_fc = pd.read_csv(fpath, index_col=0)

    # Store ROI order from the first file and enforce consistency
    if sub_rois is None:
        sub_rois = df_fc.index.tolist()
    else:
        if df_fc.index.tolist() != sub_rois or df_fc.columns.tolist() != sub_rois:
            raise ValueError(
                f"ROI ordering mismatch in file: {fpath}"
            )

    subFC_mats.append(df_fc.values)

# Convert to array: n_subjects x n_rois x n_rois
subFC_mats = np.stack(subFC_mats, axis=0)

# ------------------------------------------------------------------
# Group-average sub-FC across subjects
# ------------------------------------------------------------------
mean_FC = np.mean(subFC_mats, axis=0)
subFC = mean_FC.copy()

# ------------------------------------------------------------------
# Rebuild ROI mapping for the already-saved FC matrices
# sub_rois should match the saved ROI-restricted FC order
# ------------------------------------------------------------------
rois_incl = sub_rois
rois_map = ['LH_SomMot' if 'LH' in roi else 'RH_SomMot' if 'RH' in roi and 'SomMot' in roi else roi.split(' ')[0]
            for roi in rois_incl]

# Identify cortical SMC indices within rois_incl
sm_idx = [j for j, roi in enumerate(rois_map) if 'SomMot' in roi]

# ------------------------------------------------------------------
# Extract cortical SMC-SMC block
# ------------------------------------------------------------------
cortical_FC = subFC[np.ix_(sm_idx, sm_idx)]

# ------------------------------------------------------------------
# Sparsify cortical FC by retaining strongest edges per row
# ------------------------------------------------------------------
sparsity_cortical = 0.9
cortical_sparse = np.array([
    row * (row > np.sort(row)[int(sparsity_cortical * len(row)) - 1])
    for row in cortical_FC
])

# ------------------------------------------------------------------
# Replace cortical block in the full subFC to obtain corticospinal FC
# ------------------------------------------------------------------
FC_cs = subFC.copy()
FC_cs[np.ix_(sm_idx, sm_idx)] = cortical_sparse

## Panel A – SMC FC matrix

Plot the z-scored, sparsified SMC FC matrix (62×62), ordered by hemisphere.


In [ ]:
# Panel A: SMC-only FC matrix (cortical_FC)

# Z-score and clip to [-1, 1]
FC_z = (cortical_FC - cortical_FC.mean()) / cortical_FC.std()
FC_z = np.clip(FC_z, -1, 1)

# Diverging colormap tuned for FC visualization
colors = ["#00a2ff", "#9ddff5", "#ffffff", "#ffbfdf", "#ff369b"]
cmap_css = LinearSegmentedColormap.from_list("css_fc_div", colors, N=256)

n = FC_z.shape[0]
cell_size = 0.1  # inches per cell
fig_size = n * cell_size

fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=300)

sns.heatmap(
    FC_z,
    cmap=cmap_css,
    square=True,
    cbar=True,
    linewidths=0.25,
    linecolor='white',
    xticklabels=False,
    yticklabels=False,
    ax=ax
)

cbar = ax.collections[0].colorbar
cbar.set_label("Z-scored FC", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_panelA_FC_SMC.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel B – Gradient explained variance

Compute diffusion map embedding on the sparsified corticospinal FC and plot the proportion of variance explained by the top five components.


In [ ]:
# Panel B: diffusion map gradients and explained variance

# Fit gradients for cortical-only and corticospinal FC
grads_cortical, lambdas_cortical, FC_cortical = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0
)

grads_corticospinal, lambdas_corticospinal, FC_corticospinal = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    rois_incl_y=rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0
)

smc_grad = grads_cortical
smc_spinal_grad = grads_corticospinal

smc_grad_norm, smc_spinal_grad_aligned, disparity = align_procrustes_matlab(smc_spinal_grad, smc_grad, align_dims=2)

# project to array format with dimensions compatible for plotting in brainspace, and save in grad_arr/ folder
# this .npy file can then be loaded locally for plotting using brainspace's plot_hemispheres function
g_cortical_norm = project_brainspace_arr(smc_grad_norm[:,:2], [rois_incl[j] for j in sm_idx], fill=np.nan, nrois_schaefer=400)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_2D_norm.npy', g_cortical_norm)

g_corticospinal_aligned = project_brainspace_arr(smc_spinal_grad_aligned[:,:2], [rois_incl[j] for j in sm_idx], fill=np.nan, nrois_schaefer=400)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_spinal_2D_aligned.npy', g_corticospinal_aligned)

### Check Disparity
print(f"Disparity : grads_corticospinal, grads_cortical = {disparity:.3f}")

### Create grad dataframe
grads = {}
grads.update({
    f'SMC_Unaligned': smc_grad,
    f'SMC-spinal_Unaligned': smc_spinal_grad,
    f'SMC_Norm':smc_grad_norm, 
    f'SMC-spinal_Aligned':smc_spinal_grad_aligned
    })

for grad_type in ['SMC_Norm', 'SMC-spinal_Aligned']:
    if grads[grad_type].shape[0]!=77:
        g = np.empty((77,grads[grad_type].shape[1]))
        g[:] = np.nan
        i = 0
        for idx, label in enumerate(smc_labels):
            if label in (
                [f'7Networks_LH_SomMot_{id}' for id in range(1, 8)] +   # LH: 1–8
                [f'7Networks_RH_SomMot_{id}' for id in range(1, 9)]    # RH: 1–9
            ):
                continue
            g[idx, :] = grads[grad_type][i, :]
            i += 1
    else:
        g = grads[grad_type]

    

# Explained variance plot (cortical gradients)
x = np.arange(1, len(lambdas_cortical) + 1)

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.plot(x, lambdas_cortical, marker='o', color='k', linewidth=1)
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance (%)')
ax.axvspan(0.95, 2.05, color='grey', alpha=0.1)
ax.set_xticks(x)
ax.set_ylim(0, max(lambdas_cortical) * 1.1)
plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_panelB_explained_variance.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel C – Cortical gradient maps

Project the first two SMC gradients (G1, G2) onto the Schaefer cortical surface and save arrays for external visualization (e.g. BrainSpace).


In [ ]:
# Panel C: project G1 and G2 onto Schaefer surface

smc_grad = grads_cortical
smc_spinal_grad = grads_corticospinal

# Align spinal gradients to cortical via Procrustes
smc_grad_norm, smc_spinal_grad_aligned, disparity = align_procrustes_matlab(
    smc_spinal_grad,
    smc_grad,
    align_dims=2
)

# Project to full Schaefer400 for surface plotting (BrainSpace)
g_cortical_norm = project_brainspace_arr(
    smc_grad_norm[:, :2],
    [rois_incl[j] for j in sm_idx],
    fill=np.nan,
    nrois_schaefer=400
)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_2D_norm.npy', g_cortical_norm)

g_corticospinal_aligned = project_brainspace_arr(
    smc_spinal_grad_aligned[:, :2],
    [rois_incl[j] for j in sm_idx],
    fill=np.nan,
    nrois_schaefer=400
)
# np.save(params["save_main_smc_npy"] + 'grad_SMC_spinal_2D_aligned.npy', g_corticospinal_aligned)

print(f"Disparity (corticospinal vs cortical gradients): {disparity:.3f}")


## Panel D – G1 vs G2 scatter (no clusters)

Scatter plot of normalized SMC gradient values (G1 vs G2) across all parcels, without cluster assignment.


In [ ]:
# Panel D: z-scored G1–G2 scatter (no clustering)

# Build SMC gradient dataframe
smc_grad_df = pd.DataFrame({
    'G1': smc_grad[:, 0],
    'G2': smc_grad[:, 1],
    'RoI': [rois_incl[j] for j in sm_idx]
})

# z-score across all SMC parcels
smc_grad_df['z_G1'] = (smc_grad_df['G1'] - smc_grad_df['G1'].mean()) / smc_grad_df['G1'].std()
smc_grad_df['z_G2'] = (smc_grad_df['G2'] - smc_grad_df['G2'].mean()) / smc_grad_df['G2'].std()

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
sns.scatterplot(
    data=smc_grad_df,
    x='z_G1', y='z_G2',
    color='black',
    alpha=0.6,
    s=50,
    edgecolor='k',
    ax=ax,
    legend=False
)
ax.axhline(0, color='grey', linestyle='--', linewidth=1)
ax.axvline(0, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('z(G1)')
ax.set_ylabel('z(G2)')
ax.set_title('SMC gradients: z-scored G1 vs G2 (no clusters)')
plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_panelD_G1G2_scatter_no_clusters.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel E – Spectral clustering and silhouette scores

Compute spectral clustering on (G1, G2) for K = 2…10 and plot mean silhouette scores, plus the silhouette coefficient distribution for K = 4.


In [ ]:
# Panel E: mean silhouette vs K (SMC gradients in G1–G2 space)

K_RANGE = range(2, 10)
RANDOM_STATE = 10

smc_df = smc_grad_df[['G1', 'G2', 'RoI']].dropna().copy()
X = smc_df[['G1', 'G2']].values
X_scaled = StandardScaler().fit_transform(X)

ks = []
mean_sil = []
for k in K_RANGE:
    clusterer = SpectralClustering(
        n_clusters=k,
        affinity='rbf',
        gamma=1.0,
        random_state=RANDOM_STATE
    )
    labels = clusterer.fit_predict(X_scaled)
    sil_avg = silhouette_score(X_scaled, labels)
    ks.append(k)
    mean_sil.append(sil_avg)

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.plot(ks, mean_sil, marker='o', color='grey',
        markerfacecolor="#00a2ff", markeredgecolor="#ff369b", markersize=6)
ax.axvline(4, color='k', linestyle='--', linewidth=1.5, label='K = 4')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Mean silhouette score')
ax.set_xticks(list(K_RANGE))
ax.grid(alpha=0.2, linestyle=':')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_panelE_mean_silhouette_vs_K.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

# Silhouette coefficient distribution for K = 4
clusterer_k4 = SpectralClustering(
    n_clusters=4,
    affinity='rbf',
    gamma=1.0,
    random_state=RANDOM_STATE
)
labels_k4 = clusterer_k4.fit_predict(X_scaled)

silhouette_avg = silhouette_score(X_scaled, labels_k4)
sample_silhouette_values = silhouette_samples(X_scaled, labels_k4)

fig, ax1 = plt.subplots(figsize=(4, 6), dpi=300)
ax1.set_xlim([-0.1, 1])
ax1.set_ylim([0, len(X_scaled) + (4 + 1) * 10])

y_lower = 10
for i in range(4):
    ith_cluster_silhouette_values = sample_silhouette_values[labels_k4 == i]
    ith_cluster_silhouette_values.sort()
    size_cluster_i = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i

    ax1.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        ith_cluster_silhouette_values,
        alpha=0.8
    )

    ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax1.axvline(x=silhouette_avg, color="black", linestyle="--",
            label=f"mean silhouette = {silhouette_avg:.2f}")
ax1.set_xlabel("Silhouette coefficient")
ax1.set_ylabel("Cluster label")
ax1.set_yticks([])
ax1.set_xticks(np.linspace(-0.1, 1.0, 7))
ax1.legend(frameon=False, loc="best")
ax1.set_title("Silhouette distribution (K = 4, SMC gradients)")
plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_panelE_silhouette_K4.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel F – G1–G2 clusters and functional enrichment

This block implements the K=4 spectral clustering in gradient–gradient space, maps clusters to Brainnetome-derived somatotopic classes (LL, UL, TRUNK, FA), and visualizes:
- G1–G2 scatter with K=4 spectral clusters (per enriched class)
- Enrichment scores heatmap (clusters × somatotopic classes)
- Gradient value boxplots per enriched functional class.


### Panel F.1 – Build gradient–distance dataframe and cluster (K=4)

This cell builds `df_all_dict` for SMC_Norm and SMC-spinal_Aligned gradients, clusters SMC_Norm in (Dist, G2) space using spectral clustering with K=4, and attaches z-scored Dist, G1, G2 and cluster labels.


In [ ]:
# Dataframes and clustering

lh_labels = pd.read_csv(params["dist_file_L"])
rh_labels = pd.read_csv(params["dist_file_R"])

som_labels_L = lh_labels["roi_label"].tolist()
som_labels_R = rh_labels["roi_label"].tolist()
som_labels = som_labels_L + som_labels_R

lh_dist = lh_labels["cum_distance_mm"].tolist()
rh_dist = rh_labels["cum_distance_mm"].tolist()
dist = lh_dist + rh_dist

GRAD_TYPES = ["SMC_Norm", "SMC-spinal_Aligned"]
K = 4
RANDOM_STATE = 10

def cluster_and_analyze(df_all, grad_type, k=K, random_state=RANDOM_STATE, cluster_feature="G2"):
    x_var = "Dist"
    y_var = cluster_feature
    df_sub = df_all[[x_var, y_var, "RoI", "G1", "G2"]].dropna().copy()
    if df_sub.empty:
        return None
    X = df_sub[[x_var, y_var]].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    spectral = SpectralClustering(
        n_clusters=k,
        affinity="rbf",
        gamma=1.0,
        random_state=random_state,
    )
    labels = spectral.fit_predict(X_scaled)
    df_sub["cluster"] = labels
    label_map = dict(zip(df_sub["RoI"], df_sub["cluster"]))
    df_all = df_all.copy()
    df_all["cluster_smc"] = df_all["RoI"].map(label_map)
    df_all = df_all[~df_all["cluster_smc"].isna()].copy()
    df_all["cluster_smc"] = df_all["cluster_smc"].astype(int)
    df_all["z_Dist"] = (df_all["Dist"] - df_all["Dist"].mean()) / df_all["Dist"].std()
    df_all["z_G1"] = (df_all["G1"] - df_all["G1"].mean()) / df_all["G1"].std()
    df_all["z_G2"] = (df_all["G2"] - df_all["G2"].mean()) / df_all["G2"].std()
    return df_all

df_all_dict = {}
for grad_type in GRAD_TYPES:
    g1 = grads[grad_type][:, 0]
    g2 = grads[grad_type][:, 1]
    df_all = pd.DataFrame(
        {"G1": g1, "G2": g2, "RoI": som_labels, "Dist": dist}
    )
    df_all_dict[grad_type] = cluster_and_analyze(
        df_all,
        grad_type,
        k=K,
        random_state=RANDOM_STATE,
        cluster_feature="G2",
    )

# ---------- 1) Prepare SMC and SMC-spinal data (both hemispheres together) ----------
df_smc  = df_all_dict['SMC_Norm'].copy()
df_spin = df_all_dict['SMC-spinal_Aligned'].copy()

# Keep needed columns and inner-join on RoI to ensure same set
df_smc  = df_smc[['RoI', 'G1', 'G2', 'z_Dist']].dropna()
df_spin = df_spin[['RoI', 'G1', 'G2', 'z_Dist']].dropna()

df_smc  = df_smc.sort_values('RoI').reset_index(drop=True)
df_spin = df_spin.sort_values('RoI').reset_index(drop=True)

common_rois = sorted(set(df_smc['RoI']) & set(df_spin['RoI']))
df_smc  = df_smc[df_smc['RoI'].isin(common_rois)].sort_values('RoI').reset_index(drop=True)
df_spin = df_spin[df_spin['RoI'].isin(common_rois)].sort_values('RoI').reset_index(drop=True)

# z-score G1,G2 in SMC space for clustering (across both hemispheres)
df_smc['z_G1'] = (df_smc['G1'] - df_smc['G1'].mean()) / df_smc['G1'].std()
df_smc['z_G2'] = (df_smc['G2'] - df_smc['G2'].mean()) / df_smc['G2'].std()

# ---------- 2) Spectral clustering in SMC G1–G2 ----------
K = 4
X_smc = df_smc[['z_G1', 'z_G2']].values
spectral = SpectralClustering(
    n_clusters=K,
    affinity='rbf',
    gamma=1.0,
    random_state=10
)
cluster_ids_smc = spectral.fit_predict(X_smc)

df_smc['cluster_smc']  = cluster_ids_smc
df_spin['cluster_smc'] = df_smc['cluster_smc'].values

# ---------- 3) Map G1–G2-based clusters back onto full SMC / spinal dfs ----------
df_all_smc    = df_all_dict['SMC_Norm'].copy()
df_all_spinal = df_all_dict['SMC-spinal_Aligned'].copy()

cluster_map_smc  = dict(zip(df_smc['RoI'],  df_smc['cluster_smc']))
cluster_map_spin = dict(zip(df_spin['RoI'], df_spin['cluster_smc']))

df_all_smc['cluster_smc']    = df_all_smc['RoI'].map(cluster_map_smc)
df_all_spinal['cluster_smc'] = df_all_spinal['RoI'].map(cluster_map_spin)

# Drop RoIs without a cluster, cast to int
df_all_smc    = df_all_smc.dropna(subset=['cluster_smc']).copy()
df_all_spinal = df_all_spinal.dropna(subset=['cluster_smc']).copy()
df_all_smc['cluster_smc']    = df_all_smc['cluster_smc'].astype(int)
df_all_spinal['cluster_smc'] = df_all_spinal['cluster_smc'].astype(int)

### Panel F.2 – Functional maps, cluster lists, and enrichment scores

This cell:
- Defines the Brainnetome-derived functional maps for LH/RH SMC parcels.
- Specifies the four cluster lists (K=4) used in the manuscript.
- Builds `df_func` (RoI × cluster × functional label × coarse class) and computes enrichment scores vs global base rates (FA, UL, LL, TRUNK).


In [ ]:
# --- functional maps (same as before) ---
func_map_LH = {
    8:  "A1/2/3tonIa (tongue/lar)",
    9:  "A4tl (tongue/lar)",
    10: "A1/2/3tonIa (tongue/lar)",
    11: "A1/2/3tonIa (tongue/lar)",
    12: "A1/2/3ulhf (UL/H/F)",
    13: "A1/2/3ulhf (UL/H/F)",
    14: "A1/2/3ulhf (UL/H/F)",
    15: "A1/2/3ulhf (UL/H/F)",
    16: "A4hf (H/F)",
    17: "A1/2/3ulhf (UL/H/F)",
    18: "A4ll (LL)",
    19: "A4hf (H/F)",
    20: "A1/2/3ulhf (UL/H/F)",
    21: "A2l + A1/2/3ulhf (UL/H/F)",
    22: "A4ul (UL)",
    23: "A1/2/3ll (LL)",
    24: "A4ll (LL)",
    25: "A4ll (LL)",
    26: "A4hf (H/F)",
    27: "A1/2/3tru",
    28: "A4ul (UL)",
    29: "A2l + A1/2/3tru (TRU)",
    30: "A4ul (UL)",
    31: "A4ul (UL)",
    32: "A1/2/3ll (LL)",
    33: "A4ll (LL)",
    34: "A4t (TRU)",
    35: "A1/2/3tru (TRU)",
    36: "A1/2/3tru (TRU)",
    37: "A4t (TRU)",
}

func_map_RH = {
    9:  "A1/2/3tonIa (tongue/lar)",
    10: "A4hf (H/F)",
    11: "A1/2/3tonIa (tongue/lar)",
    12: "A4tl (tongue/lar)",
    13: "A4hf (H/F)",
    14: "A1/2/3tonIa (tongue/lar)",
    15: "A4hf (H/F)",
    16: "A4hf (H/F)",
    17: "A2r + A1/2/3tonIa (tongue/lar)",
    18: "A4hf (H/F)",
    19: "A2r + A1/2/3ulhf (UL/H/F)",
    20: "A1/2/3ll (LL)",
    21: "A1/2/3ulhf (UL/H/F)",
    22: "A4ul (UL)",
    23: "A2r + A1/2/3ulhf (UL/H/F)",
    24: "A4ll (LL)",
    25: "A4ul (UL)",
    26: "A4ul (UL)",
    27: "A2r + A1/2/3tru (TRU)",
    28: "A4ul (UL)",
    29: "A1/2/3ulhf (UL/H/F)",
    30: "A4ll (LL)",
    31: "A1/2/3tru",
    32: "A1/2/3ll (LL)",
    33: "A4t (TRU)",
    34: "A4ul (UL)",
    35: "A1/2/3tru (TRU)",
    36: "A4ul (UL)",
    37: "A1/2/3tru",
    38: "A4ll (LL)",
    39: "A4t (TRU)",
    40: "A1/2/3tru (TRU)",
}


def parse_hemi_and_index(roi_name: str):
    parts = roi_name.split('_')
    hemi = parts[1]
    idx = int(parts[-1])
    return hemi, idx


def roi_to_func_label(roi_name: str) -> str:
    hemi, idx = parse_hemi_and_index(roi_name)
    if hemi == 'LH':
        return func_map_LH.get(idx, 'Unknown')
    else:
        return func_map_RH.get(idx, 'Unknown')


# --- 4 SMC clusters as given ---

cluster0 = [
    '7Networks_LH_SomMot_37', '7Networks_LH_SomMot_26', '7Networks_LH_SomMot_19',
    '7Networks_LH_SomMot_25', '7Networks_LH_SomMot_15', '7Networks_LH_SomMot_14',
    '7Networks_LH_SomMot_13', '7Networks_LH_SomMot_10', '7Networks_LH_SomMot_9',
    '7Networks_LH_SomMot_11', '7Networks_LH_SomMot_8', '7Networks_LH_SomMot_12',
    '7Networks_RH_SomMot_40', '7Networks_RH_SomMot_21', '7Networks_RH_SomMot_11',
    '7Networks_RH_SomMot_10', '7Networks_RH_SomMot_12', '7Networks_RH_SomMot_15',
    '7Networks_RH_SomMot_13', '7Networks_RH_SomMot_17', '7Networks_RH_SomMot_18',
    '7Networks_RH_SomMot_19', '7Networks_RH_SomMot_16', '7Networks_RH_SomMot_9',
]

cluster1 = [
    '7Networks_LH_SomMot_18', '7Networks_LH_SomMot_16', '7Networks_LH_SomMot_17',
    '7Networks_LH_SomMot_20', '7Networks_RH_SomMot_20', '7Networks_RH_SomMot_22',
    '7Networks_RH_SomMot_23', '7Networks_RH_SomMot_26', '7Networks_RH_SomMot_24',
    '7Networks_RH_SomMot_30',
]

cluster2 = [
    '7Networks_LH_SomMot_21', '7Networks_LH_SomMot_23', '7Networks_LH_SomMot_22',
    '7Networks_LH_SomMot_29', '7Networks_LH_SomMot_30', '7Networks_LH_SomMot_28',
    '7Networks_LH_SomMot_36', '7Networks_RH_SomMot_28', '7Networks_RH_SomMot_29',
    '7Networks_RH_SomMot_36', '7Networks_RH_SomMot_35', '7Networks_RH_SomMot_34',
    '7Networks_RH_SomMot_39',
]

cluster3 = [
    '7Networks_LH_SomMot_32', '7Networks_LH_SomMot_31', '7Networks_LH_SomMot_35',
    '7Networks_LH_SomMot_34', '7Networks_LH_SomMot_33', '7Networks_LH_SomMot_24',
    '7Networks_LH_SomMot_27', '7Networks_RH_SomMot_31', '7Networks_RH_SomMot_25',
    '7Networks_RH_SomMot_32', '7Networks_RH_SomMot_27', '7Networks_RH_SomMot_37',
    '7Networks_RH_SomMot_33', '7Networks_RH_SomMot_38', '7Networks_RH_SomMot_14',
]

clusters = {
    0: cluster0,
    1: cluster1,
    2: cluster2,
    3: cluster3,
}

# --- Build a dataframe with RoI, cluster id, and functional label ---

rows = []
for cid, roi_list in clusters.items():
    for roi in roi_list:
        rows.append({
            "cluster_smc": cid,
            "RoI": roi,
            "func_label": roi_to_func_label(roi),
        })

df_func = pd.DataFrame(rows)

# --- Frequency map per cluster ---

for cid in sorted(clusters.keys()):
    sub = df_func[df_func["cluster_smc"] == cid]
    counts = sub["func_label"].value_counts()
    #print(f"\nCluster {cid} (n={len(sub)}):")
    #print(counts)


# --- Functional maps provided (func_map_LH, func_map_RH) ---
# (keep your existing func_map_LH, func_map_RH, parse_hemi_and_index, roi_to_func_label)

def parse_hemi_and_index(roi_name: str):
    parts = roi_name.split('_')
    hemi = parts[1]
    idx = int(parts[-1])
    return hemi, idx

def roi_to_func_label(roi_name: str) -> str:
    hemi, idx = parse_hemi_and_index(roi_name)
    if hemi == 'LH':
        return func_map_LH.get(idx, 'Unknown')
    else:
        return func_map_RH.get(idx, 'Unknown')

def func_to_class(func_label: str) -> str:
    lab = func_label.lower()

    if 'tru' in lab:
        return 'TRU'
    if 'll' in lab:
        return 'LL'
    if 'ul' in lab:
        return 'UL'
    # HF and FA both mapped to a single FA class
    if 'hf' in lab or 'h/f' in lab or 'ton' in lab or 'tongue' in lab or 'lar' in lab:
        return 'FA'
    return 'Other'


# --- 4 SMC clusters as given (keep your cluster lists) ---

clusters = {
    0: cluster0,
    1: cluster1,
    2: cluster2,
    3: cluster3,
}

# --- Build a dataframe with RoI, cluster id, functional label, and coarse class ---

rows = []
for cid, roi_list in clusters.items():
    for roi in roi_list:
        f_label = roi_to_func_label(roi)
        coarse = func_to_class(f_label)
        rows.append({
            "cluster_smc": cid,
            "RoI": roi,
            "func_label": f_label,
            "coarse_class": coarse
        })

df_func = pd.DataFrame(rows)

# --- Enrichment: assign 1 dominant class per cluster_SMC ---

cluster_to_class = {}
cluster_comp = {}

# Cluster-specific preference order, based on your interpretation
preference = {
    0: ['HF', 'FA', 'UL', 'LL', 'TRU', 'Other'],  # Cluster 0: face/arm
    1: ['LL', 'UL', 'HF', 'TRU', 'FA', 'Other'],  # Cluster 1: lower limb
    2: ['UL', 'TRU', 'LL', 'HF', 'FA', 'Other'],  # Cluster 2: upper limb-like
    3: ['TRU', 'LL', 'UL', 'HF', 'FA', 'Other'],  # Cluster 3: trunk
}

for cid in sorted(clusters.keys()):
    sub = df_func[df_func["cluster_smc"] == cid]
    counts = sub["coarse_class"].value_counts()
    cluster_comp[cid] = counts

    prefs = preference[cid]

    # among classes that appear, pick the first in the preference list
    chosen = None
    for cls in prefs:
        if cls in counts.index:
            chosen = cls
            break

    cluster_to_class[cid] = chosen

    # print(f"\nCluster {cid} (n={len(sub)}):")
    # print(counts)
    # print(f" -> Enriched class: {chosen}")


In [ ]:
from collections import Counter
# Classes of interest (order fixed)
classes = ["FA", "UL", "LL", "TRU"]

# -----------------------------
# Build df_clusters from df_func
# -----------------------------
# one row per ROI with its cluster and coarse_class
df_clusters = df_func[["RoI", "cluster_smc", "coarse_class"]].copy()
df_clusters["cluster_smc"] = df_clusters["cluster_smc"].astype(int)

# -----------------------------
# Global base rates
# -----------------------------
global_counts = Counter(df_clusters["coarse_class"].tolist())
N_global = len(df_clusters)
p_global = {cls: global_counts.get(cls, 0) / N_global for cls in classes}

# -----------------------------
# Cluster-wise proportions, enrichment, purity
# -----------------------------
rows = []

for c in sorted(df_clusters["cluster_smc"].unique()):
    df_c = df_clusters[df_clusters["cluster_smc"] == c]
    labels = df_c["coarse_class"].tolist()
    n = len(labels)
    if n == 0:
        continue

    counts = Counter(labels)
    # cluster proportions
    p_cluster = {cls: counts.get(cls, 0) / n for cls in classes}

    # enrichment = p_cluster / p_global
    enrichment = {
        cls: (p_cluster[cls] / p_global[cls]) if p_global[cls] > 0 else np.nan
        for cls in classes
    }

    # best class by enrichment
    best_cls_enrich = max(enrichment, key=lambda cls: enrichment[cls])
    best_enrich_val = enrichment[best_cls_enrich]

    # best class by purity (within-cluster proportion)
    best_cls_p = max(p_cluster, key=lambda cls: p_cluster[cls])
    best_p_val = p_cluster[best_cls_p]

    rows.append({
        "cluster": c,
        "n": n,
        "p_FA": p_cluster["FA"],
        "p_UL": p_cluster["UL"],
        "p_LL": p_cluster["LL"],
        "p_TRU": p_cluster["TRU"],
        "enrich_FA": enrichment["FA"],
        "enrich_UL": enrichment["UL"],
        "enrich_LL": enrichment["LL"],
        "enrich_TRU": enrichment["TRU"],
        "assigned_class_enrichment": best_cls_enrich,
        "enrichment_value": best_enrich_val,
        "assigned_class_purity": best_cls_p,
        "purity_value": best_p_val,
    })

df_enrich = pd.DataFrame(rows)
print(df_enrich)


### Panel F.3 – G1–G2 scatter with K=4 clusters and KDE per enriched class

This cell merges SMC_Norm gradients with functional classification, assigns each cluster to its dominant somatotopic class via enrichment, and plots G1–G2 scatter with per-cluster KDE contours colored by enriched class.


In [ ]:
# After running: df_enrich from previous code
# Map cluster → enriched class (by enrichment or purity; pick one)
cluster_to_enriched = dict(
    zip(df_enrich["cluster"], df_enrich["assigned_class_enrichment"])
)

# Add enriched_class column to df_func
df_func["enriched_class"] = df_func["cluster_smc"].map(cluster_to_enriched)
# Merge gradients with functional classification
df_grad = df_all_smc.merge(
    df_func[["RoI", "cluster_smc", "coarse_class", "enriched_class"]],
    on=["RoI", "cluster_smc"],
    how="inner"
)

df_grad["cluster_smc"] = df_grad["cluster_smc"].astype(int)
# Colors per enriched class
class_colors = {
    "FA":  "#219ebc",
    "UL":  "#023047",
    "LL":  "#ffb703",
    "TRU": "#fb8500",
    "Other": "grey"
}

# Cluster color based on its enriched class
cluster_color = {
    cid: class_colors[cls]
    for cid, cls in cluster_to_enriched.items()
}

# Number of clusters
K = 4

# Final class colors
class_colors = {
    "FA":  "#219ebc",
    "UL":  "#ffb703",
    "LL":  "#023047",
    "TRU": "#fb8500",
}

# Cluster -> enriched class (from your enrichment)
cluster_to_homu = {0: 'FA', 1: 'LL', 2: 'UL', 3: 'TRU'}

# Per-cluster color from class colors (used for both KDE and scatter)
cluster_color = {
    c: class_colors[cls]
    for c, cls in cluster_to_homu.items()
}

fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

for c in range(K):
    df_c = df_grad[df_grad['cluster_smc'] == c]
    if df_c.empty:
        continue

    # KDE per cluster (same color as scatter)
    sns.kdeplot(
        data=df_c,
        x='G1',
        y='G2',
        levels=20,
        thresh=0.05,
        color=cluster_color[c],
        fill=True,
        alpha=0.25,
        ax=ax
    )

    # Points per cluster
    sns.scatterplot(
        data=df_c,
        x='G1',
        y='G2',
        s=200,
        color=cluster_color[c],
        edgecolor='k',
        alpha=0.8,
        ax=ax,
        label=f'Cluster {c} ({cluster_to_homu[c]})',
        legend=False  # turn legend off here
    )

ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel("Gradient 1")
ax.set_ylabel("Gradient 2")
ax.set_title("SMC gradient space with KDE per enriched class")
#ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_smc"],
    "figure1_SMC_GradientScatter_K4_KDE.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


### Panel F.4 – Enrichment scores heatmap (clusters × somatotopic classes)

This cell visualizes the enrichment scores for FA, UL, LL, TRUNK across the four clusters as a heatmap.


In [ ]:
classes = ["FA", "UL", "LL", "TRU"]
DF_ENRICH_SORT = df_enrich.sort_values("cluster").reset_index(drop=True)

enrich_mat = DF_ENRICH_SORT[[f"enrich_{c}" for c in classes]].values
E_z = np.clip(enrich_mat, 0, 2.3)

colors = ["#F9F5E0", "#F5895C", "#B34ECC"]
cmap_css = LinearSegmentedColormap.from_list("css_enrich_div3", colors, N=256)

fig, ax = plt.subplots(figsize=(6, 2.5), dpi=300)

sns.heatmap(
    E_z,
    cmap=cmap_css,
    square=True,
    cbar=True,
    linewidths=0.5,
    linecolor='white',
    xticklabels=classes,
    yticklabels=DF_ENRICH_SORT["cluster"].tolist(),
    ax=ax
)

cbar = ax.collections[0].colorbar
cbar.set_label("Enrichment (cluster / global)", rotation=270, labelpad=15)
ax.set_xlabel("Somatotopic class")
ax.set_ylabel("Cluster")
plt.tight_layout()
plt.show()

out_path = os.path.join(
    params["save_main_smc"],
    "figure1_ClusterEnrichment_K4.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel G – Gradient value boxplots per enriched functional class

This cell creates boxplot/violin overlays of |z(G1)| and |z(G2)| across the four enriched somatotopic classes. Install plotnine if not already present (!pip install plotnine)

In [ ]:
from plotnine import (
    ggplot, aes, geom_point, geom_boxplot, geom_violin, geom_line,
    position_jitter, scale_color_manual, scale_fill_manual,
    xlab, ylab, coord_cartesian, theme_classic, theme
)

class_hex_map = {
    'FA': '#219ebc',
    'UL': '#023047',
    'LL': '#023047',
    'TRU': '#fb8500'
}

class_order = ['FA', 'UL', 'LL', 'TRU']


def build_df_for_grad_enriched(df_grad, grad='G1'):
    df = df_grad.copy()
    df = df[df['enriched_class'].isin(class_order)].copy()

    z_col = f'z_{grad}'
    if z_col not in df.columns:
        df[z_col] = (df[grad] - df[grad].mean()) / df[grad].std()

    df['abs_grad'] = df[z_col].abs()
    df['enriched_class'] = pd.Categorical(
        df['enriched_class'],
        categories=class_order,
        ordered=True
    )
    return df[['RoI', 'enriched_class', 'abs_grad']]


def plot_grad_by_enriched_single_map(df_grad, grad='G1', ylim=(0, 2)):
    df = build_df_for_grad_enriched(df_grad, grad=grad)

    box_width = 0.4
    jitter_width = box_width / 2.0

    p = (
        ggplot(df)
        + geom_point(
            aes(x='enriched_class', y='abs_grad', color='enriched_class'),
            size=2.0,
            stroke=0.2,
            position=position_jitter(width=jitter_width, height=0)
        )
        + geom_boxplot(
            aes(x='enriched_class', y='abs_grad', fill='enriched_class'),
            width=box_width,
            outlier_alpha=0,
            alpha=0.15,
            color='black'
        )
        + geom_violin(
            aes(x='enriched_class', y='abs_grad', fill='enriched_class'),
            trim=True,
            alpha=0.30,
            color='black'
        )
        + scale_color_manual(values=class_hex_map)
        + scale_fill_manual(values=class_hex_map)
        + xlab('Enriched class')
        + ylab(f'|z({grad})|')
        + coord_cartesian(ylim=ylim)
        + theme_classic()
        + theme(figure_size=(4, 3), dpi=300)
    )
    return p

p_g1 = plot_grad_by_enriched_single_map(df_grad, grad='G1', ylim=(0, 2))
p_g2 = plot_grad_by_enriched_single_map(df_grad, grad='G2', ylim=(0, 2))


In [ ]:
p_g1

In [ ]:
p_g2